# Detailed Analysis (Bid by Bid, Card by Card)

This is a tutorial of how to do a detailed analysis of a played board.

The engine looks at the bidding and play as it originally happened, and does an analysis for every bid and every card played.

The analysis is not just a double-dummy analysis for the exact current layout (like if you would press the "GIB" button on BBO). Instead, it's an analysis over many different possible layouts (samples).

In [1]:
import os

# Run from ben/src. Guarded so re-running this cell in the same kernel
# does not walk further up and break the imports below.
# Run from ben/src, wherever this notebook sits. Walks up to the nearest 'src'
# directory, or down into one if we started at the repository root. Idempotent,
# so re-running the cell in the same kernel is harmless.
_here = os.getcwd()
while os.path.basename(_here) != 'src' and os.path.dirname(_here) != _here:
    _here = os.path.dirname(_here)
if os.path.basename(_here) == 'src':
    os.chdir(_here)
elif os.path.isdir(os.path.join(os.getcwd(), 'src')):
    os.chdir('src')

from nn.models_tf2 import Models
from analysis import CardByCard
from util import parse_lin, display_lin
from sample import Sample
import conf
import numpy as np

np.set_printoptions(precision=2, suppress=True, linewidth=200)
np.random.seed(42)

models = Models.from_conf(conf.load('./config/default.conf'),'..')   # loading neural networks
sampler = Sample.from_conf(conf.load('./config/default.conf'), False)  # Load sampling strategies

# PIMC is turned off for analysis. analysis.py builds a BGADLL but never calls
# reset_trick()/update_trick_needed() as it replays the deal, the way game.py
# does during play, so its trick state never advances and SetupEvaluation fails
# with "PIMC SetupEvaluation failed". Card play is analysed with the neural
# network and the double-dummy solver instead.
models.pimc_use_declaring = False
models.pimc_use_defending = False


Loading config file /Users/allwynsequeira/AllwynDev/ben/src/./config/default.conf


     Score |    MP |  Cross-IMP
       420 |     5 |       5.33
       420 |     5 |       5.33
       170 |     2 |       -2.0
       -50 |     0 |      -8.67


Loading config file /Users/allwynsequeira/AllwynDev/ben/src/./config/default.conf


## Analyzing a board played on BBO

In [2]:
# copy-paste from the hand records (in lin format)

lin = 'pn|You,~~M7228oka,~~M72302cm,~~M72316sq|st||md|1S4TKHJD68QC679TKA,S35H479TQKD24TAC8,S2789H3AD379JKC35,|rh||ah|Board 3|sv|e|mb|1C|an|Minor suit opening -- 3+ !C; 11-21 HCP; 12-22 total points|mb|2H|an|Aggressive weak jump overcall -- 6+ !H; 4-10 HCP |mb|d|an|Negative double -- 4+ !S; 7+ HCP; 8+ total points |mb|4H|an|The Law: 10 trump -> game support -- 4+ total points |mb|4S|an|3+ !C; 4+ !S; 16-21 HCP; 17-22 total points|mb|p|mb|p|mb|p|pg||pc|DA|pc|D3|pc|D5|pc|D6|pg||pc|C8|pc|C3|pc|CJ|pc|CA|pg||pc|S4|pc|S5|pc|S8|pc|SJ|pg||pc|H5|pc|HJ|pc|HQ|pc|HA|pg||pc|S2|pc|SA|pc|ST|pc|S3|pg||pc|H2|pc|SK|pc|H4|pc|H3|pg||pc|D8|pc|D2|pc|DJ|pc|S6|pg||pc|SQ|pc|C6|pc|H7|pc|S7|pg||pc|H8|pc|C7|pc|HK|pc|S9|pg||pc|C5|pc|C2|pc|CT|pc|HT|pg||pc|CK|pc|H9|pc|D7|pc|C4|pg||pc|DQ|pc|D4|pc|DK|pc|H6|pg||pc|D9|pc|CQ|pc|C9|pc|DT|pg||'

In [3]:
display_lin(lin)

In [4]:
board = parse_lin(lin)
print(board)

Board(dealer='S', vuln=[False, True], hands=['9872.A3.KJ973.53', 'QAJ6.2568.5.QJ42', 'KT4.J.Q86.AKT976', '53.KQT974.AT42.8'], auction=['1C', '2H', 'X', '4H', '4S', 'PASS', 'PASS', 'PASS'], play=['DA', 'D3', 'D5', 'D6', 'C8', 'C3', 'CJ', 'CA', 'S4', 'S5', 'S8', 'SJ', 'H5', 'HJ', 'HQ', 'HA', 'S2', 'SA', 'ST', 'S3', 'H2', 'SK', 'H4', 'H3', 'D8', 'D2', 'DJ', 'S6', 'SQ', 'C6', 'H7', 'S7', 'H8', 'C7', 'HK', 'S9', 'C5', 'C2', 'CT', 'HT', 'CK', 'H9', 'D7', 'C4', 'DQ', 'D4', 'DK', 'H6', 'D9', 'CQ', 'C9', 'DT'])


In [5]:
card_by_card = CardByCard(*board, models, sampler, False)

In [6]:
await card_by_card.analyze()

analyzing the bidding
1C OK NN-value: 1.200
2H OK NN-value: 1.148
X Suggested bid from NN: CandidateBid(bid=PASS, insta_score=1.1132, expected_score=---, expected_mp=---, expected_imp=---, expected_tricks=---, adjust=---, alert=  , Explain= -- ; 13- HCP))
X NN-values:CandidateBid(bid=X   , insta_score=0.0836, expected_score=---, expected_mp=---, expected_imp=---, expected_tricks=---, adjust=---, alert=  )
4H OK NN-value: 0.612
4S Suggested bid from NN: CandidateBid(bid=PASS, insta_score=1.0817, expected_score=  146, expected_mp=---, expected_imp=---, expected_tricks= 7.94, adjust=0.00, alert=  , Explain= -- 3+!C; 5-!H; 5-!S; 11-15 HCP))
4S is not in the bids from the neural network
PASS OK NN-value: 1.197
PASS OK NN-value: 1.173
PASS OK NN-value: 1.189
analyzing opening lead
DA
DA OK
analyzing play
D3 OK
D5 Forced
D6 OK
C8 OK
C3 OK
CJ OK
CA OK
S4 ? losing: 0.41
S5 OK
S8 OK
SJ OK
H5 OK
HJ OK
HQ OK
HA OK
S2 OK
SA OK
ST Forced
S3 Follow
H2 OK
SK OK
H4 OK
H3 OK
D8 OK
D2 OK
DJ OK
S6 Forced


the engine agrees with the bidding, but didn't like something in the cardplay.

playing `S4` from hand is the first mistake. apparently this play drops almost half a trick on average.

In [7]:
card_by_card.cards['S4'].to_dict()

{'card': 'HJ',
 'who': 'IMP-calc',
 'quality': '0.4776',
 'hcp': [7.5, 8.1],
 'shape': [3.1, 4.2, 3.2, 2.4, 2.4, 6.1, 2.5, 1.9],
 'candidates': [{'card': 'HJ',
   'insta_score': 0.29,
   'expected_tricks_dd': 7.55,
   'p_make_contract': 0.0,
   'expected_score_imp': 1.15,
   'msg': ''},
  {'card': 'DQ',
   'insta_score': 0.254,
   'expected_tricks_dd': 7.47,
   'p_make_contract': 0.01,
   'expected_score_imp': 1.06,
   'msg': ''},
  {'card': 'D8',
   'insta_score': 0.099,
   'expected_tricks_dd': 7.47,
   'p_make_contract': 0.01,
   'expected_score_imp': 1.06,
   'msg': ''},
  {'card': 'CK',
   'insta_score': 0.144,
   'expected_tricks_dd': 7.41,
   'p_make_contract': 0.0,
   'expected_score_imp': 0.89,
   'msg': ''},
  {'card': 'S4',
   'insta_score': 0.043,
   'expected_tricks_dd': 7.14,
   'p_make_contract': 0.01,
   'expected_score_imp': 0.42,
   'msg': ''},
  {'card': 'ST',
   'insta_score': 0.006,
   'expected_tricks_dd': 7.13,
   'p_make_contract': 0.0,
   'expected_score_imp': 

the opening lead of `DA` is interesting. the engine prefers the `HK` and it's the only card it considers.

In [8]:
card_by_card.cards['DA'].to_dict()

{'card': 'DA',
 'who': '',
 'quality': '0.7976',
 'hcp': [4.2, 2.2, 3.6, 3.0, 2.8, 4.2, 3.2, 2.7, 4.1, 1.0, 2.2, 5.7],
 'shape': [11.1, 6.4, 13.4],
 'candidates': [{'card': 'DA',
   'insta_score': 0.042,
   'p_make_contract': 0.1,
   'expected_score_imp': 0.74,
   'msg': ''},
  {'card': 'C8',
   'insta_score': 0.365,
   'p_make_contract': 0.1,
   'expected_score_imp': 0.39,
   'msg': ''},
  {'card': 'HK',
   'insta_score': 0.523,
   'p_make_contract': 0.06,
   'expected_score_imp': 0.21,
   'msg': 'suit adjust=0.5'},
  {'card': 'Sx',
   'insta_score': 0.039,
   'p_make_contract': 0.03,
   'expected_score_imp': -0.84,
   'msg': ''}],
 'samples': ['xx.KQT9xx.ATxx.8 KQxx.x.Qxxxx.AQx Axx.Jxxxx.Jx.Jxx JTxx.A.Kx.KTxxxx - 0.73866 | HK:-450.0 C8:-420.0 DA:-420.0 Sx:-450.0',
  'xx.KQT9xx.ATxx.8 KTxx.Ax.QJx.xxxx xx.Jxxxx.Kxxx.Tx AQJxx..xx.AKQJxx - 0.88019 | HK:-480.0 C8:-480.0 DA:-450.0 Sx:-480.0',
  'xx.KQT9xx.ATxx.8 Txxx.x.Qxxx.AKxx Axx.AJxxx.xxx.xx KQJx.x.KJ.QJTxxx - 0.90147 | HK:50.0 C8:50.0